# **Безтипове лямбда-числення. _Основи_**

Цей блокнот показує, як можна розуміти лямбда-вирази з точки зору об'єктно-орієнтованої парадигми програмування.

Для цього визначаються п'ять класів:

- клас `Var`, екземплярами якого є імена атомарних сутностей і який реалізується як підклас вбудованого класу `str`;
- абстрактний клас `Term`, який призначений лише для групуівння окремих різновидів лямбда-виразів;
- клас `Atm`, екземплярами якого є атомарні вирази, що знаходяться у взаємно-однозначній відповідності із екземплярами класу `Var`;
- клас `App`, екземплярами якого є вирази, що моделюють виконання операції застосування функції до даних;
- клас `Abs`, екземплярами якого є визначення функцій з позначенням аргументу та тіла функції.

## **Імпорт та визначення модулів та фінкцій, що використовуються**

In [2]:
import typing
import re

## **Змінні (імена)**

Змінні є цеглинками для побудови лямбда-виразів (термів).

Вважаємо, що

1. можливо побудувати будь-яку скінченну множину змінних;
2. між змінними та  їх іменами є взаємно-однозначна відповідність.

Технічно це реалізується так:

3. ім'я змінної є рядком, що починається з малої латинської літери, далі він може містити будь-яку кількість латинських літер (великих і малих), десяткових цифр та символів '_';
4. кожна змінна може розміщуватися лише в одному місті пам'яті.

In [3]:
class Var(str):

    __registry: dict[str, 'Var'] = {}              # реєстр використаних імен
    __pattern = re.compile(r"[a-z][a-zA-Z0-9_]*")  # шаблон імені змінної

    @classmethod
    def checkname(cls, name: object) -> str:
        """
        Метод перевіряє, чи може об'єкт бути використаний як ім'я змінної, тобто
        1) він є рядком;
        2) рядок має необхідний формат.

        Метод викидає
            виключення TypeError, якщо не виконана умова 1) та
            виключення ValueError, якщо не виконана умова 2).

        У разі виконання обох умов метод повертає name.
        """
        if not isinstance(name, str):
            # Помилка: тип 'name' не є 'str'
            raise TypeError(
                f"Type of 'name' must be 'str' but obtained {type(name)}")
        if not cls.__pattern.fullmatch(name):
            # Помилка: 'name' не відповідати шаблону 'Var.__pattern'
            raise ValueError("Invalid 'name' format")
        return name

    def __new__(cls, name: str) -> 'Var':
        """
        Метод створює та реєструє змінну з ім'ям name, якщо такої змінної ще
        немає в реєстрі, та повертає її.
        Метод повертає змінну з ім'ям name з реєстру, якщо вона там є.

        Метод може викинути виключення при виконання cls.checkname(name).
        """
        if cls.checkname(name) not in cls.__registry:
            # Створення та реєстрація в реєстрі 'Var.__registry' змінної на ім'я
            # name, якщо така змінна ще не зареєстрована
            cls.__registry[name] = super().__new__(cls, name)
        return cls.__registry[name]

    @classmethod
    def _show_registry(cls) -> dict[str, 'Var']:
        """
        Технічний метод для контролю реєстру - повертає словник реєстру.
        """
        return cls.__registry

    @classmethod
    def fresh(cls, base: str = "x") -> 'Var':
        """
        Метод створює абсолютно 'свіжу' (таку, яка ще не використовувалася)
        змінну з ім'ям виду 'base + _ + str(i)', де i є версією - цілим не
        меншим за 1.

        Метод повертає створену змінну.
        """
        # Перевіряємо базу, якщо вона "бита" — встановлюємо її як "x"
        try:
            cls.checkname(base)
        except (TypeError, ValueError):
            base = "x"
        # Якщо ім'я base ще не використано — беремо його
        if base not in cls.__registry:
            return cls(base)
        # Інакше додаємо до нього ще неіснуючий суфікс _1, _2, ...
        i = 1
        while True:
            candidate = f"{base}_{i}"
            if candidate not in cls.__registry:
                return cls(candidate)
            i += 1

Приклади реєстрації змінної

In [29]:
for name in ['x', 1, 'y_1', '_x', 'zZ', '1x']:
    print("Змінну на ім'я", end=" ")
    try:
        _ = Var(name)
        print(f"'{name}' зареєстровано.")
    except TypeError:
        print(f"{name} відхилено через помилку типу.")
    except ValueError:
        print(f"'{name}' відхилено через помилку формату.")
print(f"Реєстр: {Var._show_registry()}")

Змінну на ім'я 'x' зареєстровано.
Змінну на ім'я 1 відхилено через помилку типу.
Змінну на ім'я 'y_1' зареєстровано.
Змінну на ім'я '_x' відхилено через помилку формату.
Змінну на ім'я 'zZ' зареєстровано.
Змінну на ім'я '1x' відхилено через помилку формату.
Реєстр: {'x': 'x', 'y_1': 'y_1', 'zZ': 'zZ'}


## **Лямбда-вирази або терми**

Лямбда-вирази (терми) є громадянами першого класу лямбда-числення.  
Синтаксично, вони конструюються у відповідності до такої граматики

```
term ::= atom | application | abstraction
atom ::= Atm(name)
application ::= App(term, term)
abstraction ::= Abs(name, term)

name ::= [a-z][a-zA-Z0-9_]*
```

### **Абстрактний клас `Term`**

Цей клас є родовим для всіх всіх видів лямбда-виразів: атомів (atom), застосувань (application) та функціональних абстракцій (abstraction).

In [4]:
class Term:
    """Абстрактний клас, від якого успадковуються усі різновиди термів."""

    __slots__ = ()

    def __str__(self) -> str:
        """перетворює терм на рядок."""
        raise NotImplementedError

### **Клас атомарних виразів `Atm`**

Вирази цього виду є атомами (безструктурними константами) лямбда-числення.  
Такий вираз однозначно визначається своїм ім'ям - змінною.

In [5]:
class Atm(Term):
    """Клас представляє терми-атоми.
    Кожен екземпляр класу має єдиний атрибут
        _name: str - ім'я відповідної змінної.
    """

    __slots__ = ('_name', )

    def __init__(self, name: str):
        """Метод ініціалізує терм-атом, що відповідає змінній з ім'ям 'name'.
        """
        # Реєструємо змінну з ім'ям name, якщо такої ще немає.
        # Тут можливе виключення для некоректного name
        name = Var.checkname(name)
        self._name = name

    def __str__(self) -> str:
        return self._name

### **Клас виразів-застосувань `App`**

Вирази цього виду формалізують фразу "застосувати функцію, представлену термом $f$ до значення представленого термом $v$".

In [6]:
class App(Term):
    """Клас представляє терми-застосування.
    Кожен екземпляр класу має атрибути
        '_fun' - терм, що застосовується
        '_val' - терм, до якого застосовується
    """

    __slots__ = ('_fun', '_val')

    def __init__(self, f: Term, v: Term):
        """Метод ініціалізує терм-застосування, що визначається як результат
        застосування терму 'f' до терму 'v'.
        """
        if not (isinstance(f, Term) and isinstance(v, Term)):
            # Помилка, обидва аргументи повинні мати тип 'Term'
            raise TypeError("Both arguments must be 'Term'")
        self._fun = f
        self._val = v

    def __str__(self) -> str:
        return f"({str(self._fun)} {str(self._val)})"

### **Клас функціональних абстракцій `Abs`**

Вирази цього виду моделюють функції з визначеним аргументом, який є змінною із заданеим іменем, та термом - тілом, що залежить від аргументу.

In [7]:
class Abs(Term):
    """Клас представляє терми-абстракції.
    Кожен екземпляр класу має атрибути
        '_name' - ім'я змінної - аргументу абстракції
        '_body' - терм, що представляє тіло абстракції
    """

    __slots__ = ('_name', '_body')

    def __init__(self, x: str, t: Term):
        """Метод ініціалізує терм-абстракцію, що визначається як результат
        абстрагування за
          змінною з ім'ям x та
          термом t.
        """
        if not isinstance(t, Term):
            raise TypeError("Type of 'body' must be 'Term'")
        self._body = t
        self._name = Var(x)

    def __str__(self) -> str:
        return f"(λ {self._name}. {self._body})"


Приклади термів: $I=\lambda\ x.x$, $K=\lambda\ x.\lambda\ y.x$, $S=\lambda\ x.\lambda\ y.\lambda\ z.x\,z\,(y\,z)$

In [34]:
I, K = Abs('x', Atm('x')), Abs('x', Abs('y', Atm('x')))
S = Abs('x',
        Abs('y',
            Abs('z',
                App(
                    App(Atm('x'), Atm('z')),
                    App(Atm('y'), Atm('z'))))))
print(f"I = {I}, K = {K}, S = {S}")

I = (λ x. x), K = (λ x. (λ y. x)), S = (λ x. (λ y. (λ z. ((x z) (y z)))))


### **Вправа**

Побудуйте лямбда-вирази:

1.   $[0]=\lambda\ x.\lambda\ y.y$,
2.   $[n+1]=\lambda\ x.\lambda\ y.x\,([n]\,x\,y)$ для натурального $n$,

якщо $n=0,1,\ldots,10$.

In [8]:
def church(n: int) -> Term:
    """Повертає n-ий нумерал Черча як лямбда-вираз.

    Нумерали Черча кодують натуральні числа в лямбда-численні.
    Число n представляється функцією двох аргументів (x, y),
    яка застосовує x до y рівно n разів:
        [0]   = λ x. λ y. y
        [n+1] = λ x. λ y. x ([n] x y)

    Аргумент n — невід'ємне ціле число.
    """
    # Захист від некоректного вводу: від'ємні числа не мають нумералів
    if n < 0:
        raise ValueError("n must be a non-negative integer")

    # Базовий випадок: [0] = λ x. λ y. y
    # Нуль — це функція, яка ігнорує x і просто повертає y,
    # тобто x застосовується 0 разів
    if n == 0:
        return Abs('x', Abs('y', Atm('y')))
    else:
        # Рекурсивний крок: будуємо попередній нумерал [n-1]
        prev = church(n - 1)

        # Конструюємо аплікацію ([n-1] x y):
        # prev застосовується спочатку до x, потім результат — до y
        applied_prev = App(App(prev, Atm('x')), Atm('y'))

        # Огортаємо ще одним застосуванням x зовні:
        # x ([n-1] x y) — це додає одне застосування x
        body = App(Atm('x'), applied_prev)

        # Фінальна абстракція: λ x. λ y. body
        return Abs('x', Abs('y', body))


# Побудова та друк перших 10 нумералів Черча: [0], [1], ..., [9]
for i in range(10):
    # Для кожного i будуємо відповідний нумерал і друкуємо його
    print(f"[{i}] = {church(i)}")

[0] = (λ x. (λ y. y))
[1] = (λ x. (λ y. (x (((λ x. (λ y. y)) x) y))))
[2] = (λ x. (λ y. (x (((λ x. (λ y. (x (((λ x. (λ y. y)) x) y)))) x) y))))
[3] = (λ x. (λ y. (x (((λ x. (λ y. (x (((λ x. (λ y. (x (((λ x. (λ y. y)) x) y)))) x) y)))) x) y))))
[4] = (λ x. (λ y. (x (((λ x. (λ y. (x (((λ x. (λ y. (x (((λ x. (λ y. (x (((λ x. (λ y. y)) x) y)))) x) y)))) x) y)))) x) y))))
[5] = (λ x. (λ y. (x (((λ x. (λ y. (x (((λ x. (λ y. (x (((λ x. (λ y. (x (((λ x. (λ y. (x (((λ x. (λ y. y)) x) y)))) x) y)))) x) y)))) x) y)))) x) y))))
[6] = (λ x. (λ y. (x (((λ x. (λ y. (x (((λ x. (λ y. (x (((λ x. (λ y. (x (((λ x. (λ y. (x (((λ x. (λ y. (x (((λ x. (λ y. y)) x) y)))) x) y)))) x) y)))) x) y)))) x) y)))) x) y))))
[7] = (λ x. (λ y. (x (((λ x. (λ y. (x (((λ x. (λ y. (x (((λ x. (λ y. (x (((λ x. (λ y. (x (((λ x. (λ y. (x (((λ x. (λ y. (x (((λ x. (λ y. y)) x) y)))) x) y)))) x) y)))) x) y)))) x) y)))) x) y)))) x) y))))
[8] = (λ x. (λ y. (x (((λ x. (λ y. (x (((λ x. (λ y. (x (((λ x. (λ y. (x (((λ x. (λ y. (x (((λ x.